In [1]:
import ray
from ray import train, tune, workflow
from ray.tune import Tuner
from ray.train.xgboost import XGBoostTrainer
from ray.air.integrations.mlflow import MLflowLoggerCallback

In [ ]:
#ssh eautenrieth@10.0.7.60
# ray start --head --port=8080

# 10.0.2.15:6379 ## lokal
# ray://10.0.7.60:8080 ##  vm
# ray.init("ray://10.0.7.60:6379")
# ray.init("ray://localhost:6379")

# ssh -v -L 6379:localhost:6379 eautenrieth@10.0.7.60
# netstat -tuln | grep 6379


# ssh -v -L 10001:localhost:10001 eautenrieth@10.0.7.60
# ray start --head --ray-client-server-port=10001

# pyhton 3.10.12

In [2]:
ray.init("ray://localhost:10001")

SIGTERM handler is not set because current thread is not the main thread.


Python version:,3.10.12
Ray version:,2.8.0


In [3]:
nodes = ray.nodes()
for node in nodes:
    print(f"Node ID: {node['NodeID']}")
    print("Resources:")
    for resource_name, resource_value in node["Resources"].items():
        if resource_name == "CPU":
            print(f"  {resource_name}: {resource_value} cores")
        elif resource_name == "memory" or resource_name == "object_store_memory":
            gb_value = resource_value / (1024 ** 3)
            print(f"  {resource_name}: {gb_value:.2f} GB")
        elif "GPU" in resource_name:
            print(f"  {resource_name}: {resource_value} units")
        else:
            # Andere Ressourcen
            print(f"  {resource_name}: {resource_value}")


Node ID: 6b8a7b3d63696e09daf4534c93fc752c157399e0cd36f05daee599f0
Resources:
  node:10.0.7.60: 1.0
  CPU: 16.0 cores
  object_store_memory: 18.20 GB
  memory: 36.40 GB
  node:__internal_head__: 1.0


In [4]:
#import wandb
#from ray.air.integrations.wandb import WandbLoggerCallback
#import mlflow
#from ray.air.integrations.mlflow import MLflowLoggerCallback

In [5]:
@ray.remote(num_cpus=1)
def prepocess_data(bucket="s3://anonymous@air-example-data/breast_cancer.csv"):
    dataset = ray.data.read_csv(bucket)
    train_dataset, test_dataset = dataset.train_test_split(test_size=0.3)
    return train_dataset, test_dataset


@ray.remote(num_cpus=12, num_gpus=2)
def train_tune_model(train_dataset, test_dataset):
    
    trainer = XGBoostTrainer(
        label_column="target",
        params={
            "objective": "binary:logistic",
            "eval_metric": ["logloss", "error"],
            },
        datasets={"train": train_dataset, "test": test_dataset}, 
    )

    config = {
        "objective": "binary:logistic",
        "eval_metric": ["logloss", "error"],
        "max_depth": tune.randint(1, 9),
        "min_child_weight": tune.choice([1, 2, 3]),
        "subsample": tune.uniform(0.5, 1.0),
        "eta": tune.loguniform(1e-4, 1e-1),
    }

    tuner = Tuner(
            trainer,
            param_space={"params":config},
            tune_config=tune.TuneConfig(
                metric="test-error",
                mode="min",
                num_samples=20,
            ),
             run_config=train.RunConfig(
             callbacks=[MLflowLoggerCallback(experiment_name="ray-ml",save_artifact=True)]
         ),
    )
    result_grid = tuner.fit()
    best_result = result_grid.get_best_result(metric="test-error", mode="min")
    
    return best_result

In [ ]:
train_dataset, test_dataset = prepocess_data.bind()
model = train_tune_model.bind(train_dataset, test_dataset)

workflow.run(model)

In [6]:
train_dataset, test_dataset = ray.get(prepocess_data.remote())
best_model = ray.get(train_tune_model.remote(train_dataset, test_dataset))

(prepocess_data pid=2866607) Using autodetected parallelism=32 for stage ReadCSV to satisfy parallelism at least twice the available number of CPUs (16).
(prepocess_data pid=2866607) To satisfy the requested parallelism of 32, each read task output is split into 32 smaller blocks.
(prepocess_data pid=2866607) Using autodetected parallelism=32 for stage ReadCSV to satisfy parallelism at least twice the available number of CPUs (16).
(prepocess_data pid=2866607) To satisfy the requested parallelism of 32, each read task output is split into 32 smaller blocks.


(prepocess_data pid=2866607) [dataset]: Run `pip install tqdm` to enable progress reporting.


(_execute_read_task_split pid=2866606) /home/eautenrieth/.local/lib/python3.10/site-packages/ray/data/_internal/arrow_block.py:128: FutureWarning: promote has been superseded by mode='default'.
(_execute_read_task_split pid=2866606)   return transform_pyarrow.concat(tables)
(train_model pid=2867128) [output] This will use the new output engine with verbosity 1. To disable the new output and use the legacy output engine, set the environment variable RAY_AIR_NEW_OUTPUT=0. For more information, please see https://github.com/ray-project/ray/issues/36949


(train_model pid=2867128) ╭───────────────────────────────────────────────────────────────────────╮
(train_model pid=2867128) │ Configuration for experiment     XGBoostTrainer_2023-11-14_17-33-40   │
(train_model pid=2867128) ├───────────────────────────────────────────────────────────────────────┤
(train_model pid=2867128) │ Search algorithm                 BasicVariantGenerator                │
(train_model pid=2867128) │ Scheduler                        FIFOScheduler                        │
(train_model pid=2867128) │ Number of trials                 20                                   │
(train_model pid=2867128) ╰───────────────────────────────────────────────────────────────────────╯
(train_model pid=2867128) 
(train_model pid=2867128) View detailed results here: /home/eautenrieth/ray_results/XGBoostTrainer_2023-11-14_17-33-40
(train_model pid=2867128) To visualize your results with TensorBoard, run: `tensorboard --logdir /home/eautenrieth/ray_results/XGBoostTrainer_2023-11-14_1

(XGBoostTrainer pid=2867519) /home/eautenrieth/.local/lib/python3.10/site-packages/xgboost_ray/main.py:519: UserWarning: `num_actors` in `ray_params` is smaller than 2 (1). XGBoost will NOT be distributed!
(XGBoostTrainer pid=2867519)   warnings.warn(
(XGBoostTrainer pid=2867513) /home/eautenrieth/.local/lib/python3.10/site-packages/xgboost_ray/main.py:519: UserWarning: `num_actors` in `ray_params` is smaller than 2 (1). XGBoost will NOT be distributed!
(XGBoostTrainer pid=2867513)   warnings.warn(
(XGBoostTrainer pid=2867520) /home/eautenrieth/.local/lib/python3.10/site-packages/xgboost_ray/main.py:519: UserWarning: `num_actors` in `ray_params` is smaller than 2 (1). XGBoost will NOT be distributed!
(XGBoostTrainer pid=2867520)   warnings.warn(
(XGBoostTrainer pid=2867518) /home/eautenrieth/.local/lib/python3.10/site-packages/xgboost_ray/main.py:519: UserWarning: `num_actors` in `ray_params` is smaller than 2 (1). XGBoost will NOT be distributed!
(XGBoostTrainer pid=2867518)   warning

...

(XGBoostTrainer pid=2871987) [RayXGBoost] Finished XGBoost training on training data with total N=398 in 3.25 seconds (1.35 pure XGBoost training time).
(XGBoostTrainer pid=2871987) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/eautenrieth/ray_results/XGBoostTrainer_2023-11-14_17-33-40/XGBoostTrainer_915fe_00017_17_eta=0.0006,max_depth=8,min_child_weight=1,subsample=0.8391_2023-11-14_17-33-42/checkpoint_000000)
(XGBoostTrainer pid=2872063) [RayXGBoost] Finished XGBoost training on training data with total N=398 in 3.18 seconds (1.29 pure XGBoost training time).
(XGBoostTrainer pid=2872063) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/eautenrieth/ray_results/XGBoostTrainer_2023-11-14_17-33-40/XGBoostTrainer_915fe_00018_18_eta=0.0190,max_depth=7,min_child_weight=1,subsample=0.9110_2023-11-14_17-33-42/checkpoint_000000)
(_RemoteRayXGBoostActor pid=2872760) /home/eautenrieth/.local/lib/python3.10/site-packages/ray/data/_internal/

(train_model pid=2867128) 
(train_model pid=2867128) Trial XGBoostTrainer_915fe_00018 completed after 11 iterations at 2023-11-14 17:33:57. Total running time: 17s
(train_model pid=2867128) ╭───────────────────────────────────────────────────────────────╮
(train_model pid=2867128) │ Trial XGBoostTrainer_915fe_00018 result                       │
(train_model pid=2867128) ├───────────────────────────────────────────────────────────────┤
(train_model pid=2867128) │ checkpoint_dir_name                         checkpoint_000000 │
(train_model pid=2867128) │ time_this_iter_s                                      0.06021 │
(train_model pid=2867128) │ time_total_s                                          3.15432 │
(train_model pid=2867128) │ training_iteration                                         11 │
(train_model pid=2867128) │ test-error                                            0.05263 │
(train_model pid=2867128) │ test-logloss                                           0.5244 │
(train_m

(_RemoteRayXGBoostActor pid=2872760) [17:33:57] task [xgboost.ray]:139664666008576 got new rank 0


(train_model pid=2867128) 
(train_model pid=2867128) Trial XGBoostTrainer_915fe_00019 completed after 11 iterations at 2023-11-14 17:33:59. Total running time: 19s
(train_model pid=2867128) ╭───────────────────────────────────────────────────────────────╮
(train_model pid=2867128) │ Trial XGBoostTrainer_915fe_00019 result                       │
(train_model pid=2867128) ├───────────────────────────────────────────────────────────────┤
(train_model pid=2867128) │ checkpoint_dir_name                         checkpoint_000000 │
(train_model pid=2867128) │ time_this_iter_s                                      0.05183 │
(train_model pid=2867128) │ time_total_s                                          4.12558 │
(train_model pid=2867128) │ training_iteration                                         11 │
(train_model pid=2867128) │ test-error                                            0.04094 │
(train_model pid=2867128) │ test-logloss                                           0.2978 │
(train_m

(XGBoostTrainer pid=2872371) [RayXGBoost] Finished XGBoost training on training data with total N=398 in 4.13 seconds (2.22 pure XGBoost training time).
(XGBoostTrainer pid=2872371) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/eautenrieth/ray_results/XGBoostTrainer_2023-11-14_17-33-40/XGBoostTrainer_915fe_00019_19_eta=0.0918,max_depth=8,min_child_weight=1,subsample=0.9164_2023-11-14_17-33-42/checkpoint_000000)


In [7]:
print(f"The best model's accuracy on the test dataset is: {1 - best_model.metrics['test-error']:.2%}")

The best model's accuracy on the test dataset is: 97.08%
